# Customer evaluation: intake to final holdout

This notebook uses the same runner as `scripts/run_optimization.py`. Start with the simulation; it makes no model or AWS calls. Codex is the default agent harness in the example configuration. Read `INTEGRATION.md` for customer tools, live execution and AgentCore deployment.


In [ ]:
from pathlib import Path
from customer_eval.contract import load
from customer_eval.runner import Runner

CONFIG = Path("examples/customer-support/input.yaml")
BACKEND = "simulation"  # simulation, local, or agentcore
ALLOW_LIVE = False
OUTPUT = Path("runs/notebook-support")
RUNTIME_ARN = None
PROFILE = "YOUR_AWS_PROFILE"
REGION = "us-east-1"

if BACKEND != "simulation" and not ALLOW_LIVE:
    raise ValueError("Review routes and budget, then explicitly enable ALLOW_LIVE")
config = load(CONFIG, live=ALLOW_LIVE)
OUTPUT = OUTPUT / (BACKEND + "-" + config["fingerprint"][:12])
print({"harness": config["harness"], "backend": BACKEND, "fingerprint": config["fingerprint"]})


## Inspect the input

The policy is immutable. The optimizer can edit only the starting system prompt. Case inputs become user turns. Expected labels remain with the evaluator. Development feedback proposes changes, validation chooses them, and holdout is used after selection.


In [ ]:
print("Protected policy:\n", config["policy"])
print("Starting prompt:\n", config["prompt"])
print("Checks:", config["grading"])
print("Splits:", config["splits"])


## Run the shared workflow

Repeated commands reuse completed trials. A changed configuration or implementation needs a new output directory. Interrupted paid requests are not retried automatically.


In [ ]:
runner = Runner(config, OUTPUT, backend=BACKEND, runtime_arn=RUNTIME_ARN, profile=PROFILE, region=REGION)
report = runner.run()
print((OUTPUT / "report.md").read_text())


## Read one complete trial

For a live run, inspect the requested/returned model, conversation outputs, MCP tool arguments/results, traces, latency and grading verdicts. The simulation intentionally has a simple seed defect; its apparent gain is not evidence about Codex or either model.


In [ ]:
trial = report["holdout"]["target_selected"]["trials"][0]
print("Case:", trial["case_id"], "Status:", trial["result"]["status"])
print("Outputs:", trial["result"]["outputs"])
print("Tool calls:", trial["result"].get("tool_calls", []))
print("Checks:", trial["checks"])
print("Full artifacts:", OUTPUT.resolve())


## Use customer inputs or run live

Copy the example input directory and replace the prompt, policy, examples and checks. Use `runner.mode: prompt` / `workload: none` for text-only tasks. Use `agents[].harness: strands` for the optional Strands adapter. Live agent tools must be registered mock adapters.

For the authorized ADE test, use `examples/customer-support/ade-live.yaml`, set `EVAL_BEDROCK_BASE_URL`, select `BACKEND = "local"`, and explicitly enable `ALLOW_LIVE`. AgentCore additionally requires a deployed, qualified Runtime ARN. The selected ADE role currently has an organization-policy block on AgentCore discovery; do not treat local results as a deployed AgentCore result.


## Evaluate every turn and its tool evidence

The simulation above checks the runner plumbing. For the fuller customer example, use `examples/retail-returns/input.yaml` and [CUSTOMER-WALKTHROUGH.md](CUSTOMER-WALKTHROUGH.md). It has 8 development, 10 validation and 12 holdout conversations with clarification, corrections, evidence reuse, retry behavior and attempted policy overrides. Synthetic labels remain unreviewed by a human.

`turn_trace` verifies complete receipts; `turn_response` grades each JSON answer; `turn_tools` checks exact arguments and supporting result fields. Failed executions stay in the denominator. Both Codex and Strands emit the same turn record. The old simulation does not measure these model behaviors; use an authorized local or AgentCore run. The cell below only inspects a development case and makes no model calls.


In [ ]:
turn_config = load(Path("examples/retail-returns/input.yaml"))
example = next(case for case in turn_config["datasets"]["development"] if case["category"] == "correction")
for index, (user, expected) in enumerate(zip(example["turns"], example["expected"]["turns"]), 1):
    print(f"Turn {index}: {user}")
    print("Expected response:", expected["response"])
    print("Required tool evidence:", expected["tools"]["required"])


For a completed live run, inspect `report["holdout"]["target_selected"]["turn_quality"]` for passing/planned turns and per-case totals. Inspect each trial's `checks` and `result.turn_records` for the specific response and tool evidence. `joint_comparisons` contains fresh matched validation results. Read `selected-prompt.json` before describing a candidate as selected: the original prompt can remain the correct selection when a proposed revision does not improve the agreed tradeoff.


### Measured feedback and latency
The retail example uses a full editable operating prompt plus fixed business policy. Mutation receives development-only quality, token, cost, timing and previous-candidate evidence. `optimizer_requests` records the exact requests for review. Validation selects; holdout is evaluated only after the prompt is frozen.

Primary latency is active conversation time. Inspect each score’s `timing_metrics` for setup, teardown, provider, authentication and first-output measurements. Provider timing includes network and provider processing. Keep model migration comparisons separate from prompt comparisons. See CUSTOMER-WALKTHROUGH.md for the end-to-end customer flow.
